# Customer Churn Prediction – Machine Learning Project

This notebook builds an end-to-end machine learning pipeline to predict customer churn in the telecom domain.  
The focus is on business-driven evaluation, prioritizing recall to minimize missed churn customers.



In [38]:
import pandas as pd
import numpy as np

In [39]:
df = pd.read_csv('/content/data/WA_Fn-UseC_-Telco-Customer-Churn (1).csv')

In [40]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [41]:
df.drop(columns=['customerID'],inplace=True)

In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [43]:
df.shape

(7043, 20)

In [44]:
(df['TotalCharges'] == " ").sum()


np.int64(11)

In [45]:
# TotalCharges has empty strings for customers with tenure = 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop rows with undefined billing (very few, tenure = 0)
df = df.dropna(subset=['TotalCharges'])


In [46]:
df.isna().sum()

,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0
OnlineBackup,0


In [47]:
df = df.dropna(subset=['TotalCharges'])


In [48]:
df['Churn'].value_counts(normalize=True) * 100


,proportion
Churn,
No,73.421502
Yes,26.578498


In [49]:
pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100


Churn,No,Yes
Contract,,
Month-to-month,57.290323,42.709677
One year,88.722826,11.277174
Two year,97.151335,2.848665


In [50]:
df.groupby('Churn')['tenure'].describe()


,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5163.0,37.650010,24.076940,1.0,15.0,38.0,61.0,72.0
Yes,1869.0,17.979133,19.531123,1.0,2.0,10.0,29.0,72.0


In [51]:
# Target variable
y = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop('Churn', axis=1)


In [52]:
y.value_counts()


,count
Churn,
0,5163
1,1869


In [53]:
X_encoded = pd.get_dummies(X, drop_first=True)
feature_columns = X_encoded.columns
print("Before encoding:", X.shape)
print("After encoding:", X_encoded.shape)


Before encoding: (7032, 19)
After encoding: (7032, 30)


In [54]:
import joblib
joblib.dump(feature_columns, 'model_features.pkl')


['model_features.pkl']

In [55]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_encoded,y,test_size=0.2,random_state=42,stratify=y)


In [56]:
from sklearn.linear_model import LogisticRegression
Lr = LogisticRegression()
Lr.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [57]:
y_pred=Lr.predict(X_test)

In [58]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[911 122]
 [163 211]]
              precision    recall  f1-score   support

           0       0.85      0.88      0.86      1033
           1       0.63      0.56      0.60       374

    accuracy                           0.80      1407
   macro avg       0.74      0.72      0.73      1407
weighted avg       0.79      0.80      0.79      1407



In [59]:
y_pred_proba = Lr.predict_proba(X_test)[:, 1]
y_pred_30 = (y_pred_proba >= 0.3).astype(int)
confusion_matrix(y_test, y_pred_30)
classification_report(y_test, y_pred_30)

'              precision    recall  f1-score   support\n\n           0       0.89      0.75      0.81      1033\n           1       0.51      0.74      0.61       374\n\n    accuracy                           0.75      1407\n   macro avg       0.70      0.74      0.71      1407\nweighted avg       0.79      0.75      0.76      1407\n'

In [60]:
lr_bal = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

lr_bal.fit(X_train, y_train)
y_pred_bal = lr_bal.predict(X_test)
print(confusion_matrix(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal))

[[726 307]
 [ 76 298]]
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.80      0.73      0.74      1407



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [61]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    class_weight='balanced'
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


[[710 323]
 [ 79 295]]
              precision    recall  f1-score   support

           0       0.90      0.69      0.78      1033
           1       0.48      0.79      0.59       374

    accuracy                           0.71      1407
   macro avg       0.69      0.74      0.69      1407
weighted avg       0.79      0.71      0.73      1407



In [62]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=30,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


[[746 287]
 [ 71 303]]
              precision    recall  f1-score   support

           0       0.91      0.72      0.81      1033
           1       0.51      0.81      0.63       374

    accuracy                           0.75      1407
   macro avg       0.71      0.77      0.72      1407
weighted avg       0.81      0.75      0.76      1407



## Final Model Selection

Based on recall performance and interpretability,  
**Logistic Regression with class_weight='balanced'** was selected as the final model for deployment.


In [67]:
final_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
final_model.fit(X_train, y_train)

joblib.dump(final_model, 'churn_model.pkl')


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


['churn_model.pkl']

In [68]:
def predict_churn(input_df, model, feature_columns):
    input_encoded = pd.get_dummies(input_df)
    input_encoded = input_encoded.reindex(columns=feature_columns, fill_value=0)
    prob = model.predict_proba(input_encoded)[:, 1]
    return prob
